# 04 · Gold: business-ready tables
Star schema for the dashboard and Genie: one dimension (water systems),
one fact table (violations 2021–2025) and one county summary.

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.sdwa_gold.dim_water_system
COMMENT 'One row per Texas public water system, with type, owner, population served and primary county.'
AS
SELECT
  s.pwsid, s.system_name, s.system_type, s.owner_type, s.water_source,
  s.population_served, s.is_active, s.city,
  COALESCE(c.county, 'Unknown county') AS county
FROM workspace.sdwa_silver.water_systems s
LEFT JOIN (
  SELECT pwsid, MIN(county) AS county      -- 4 systems serve more than one county
  FROM workspace.sdwa_silver.system_counties
  GROUP BY pwsid
) c ON s.pwsid = c.pwsid

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.sdwa_gold.fact_violations
COMMENT 'One row per drinking water violation in Texas whose non-compliance period began between 2021 and 2025.'
AS
SELECT
  v.pwsid,
  v.violation_id,
  YEAR(v.noncompliance_begin_date)              AS violation_year,
  v.noncompliance_begin_date,
  v.returned_to_compliance_date,
  v.violation_category_code,
  CASE v.violation_category_code
    WHEN 'MCL'  THEN 'Contaminant limit exceeded'
    WHEN 'MRDL' THEN 'Disinfectant limit exceeded'
    WHEN 'TT'   THEN 'Treatment technique'
    WHEN 'MR'   THEN 'Monitoring and reporting'
    WHEN 'MON'  THEN 'Monitoring'
    ELSE 'Other'
  END                                           AS violation_category,
  v.is_health_based,
  v.violation_status,
  v.contaminant_code,
  CASE WHEN v.returned_to_compliance_date >= v.noncompliance_begin_date
       THEN DATEDIFF(v.returned_to_compliance_date, v.noncompliance_begin_date)
  END                                           AS days_to_compliance,
  d.county,
  d.system_name,
  d.system_type,
  d.population_served,
  d.is_active                                   AS system_is_active
FROM workspace.sdwa_silver.violations v
JOIN workspace.sdwa_gold.dim_water_system d ON v.pwsid = d.pwsid
WHERE v.noncompliance_begin_date BETWEEN '2021-01-01' AND '2025-12-31'

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.sdwa_gold.county_summary
COMMENT 'Compliance indicators per Texas county for active public water systems, violations 2021-2025.'
AS
WITH systems AS (
  SELECT county,
         COUNT(*) AS active_systems,
         SUM(CASE WHEN system_type = 'Community water system' THEN population_served END) AS community_population_served
  FROM workspace.sdwa_gold.dim_water_system
  WHERE is_active
  GROUP BY county
),
violations AS (
  SELECT county,
         COUNT(*)                                                   AS violations,
         COUNT_IF(is_health_based)                                  AS health_based_violations,
         COUNT(DISTINCT CASE WHEN is_health_based THEN pwsid END)   AS systems_with_health_based_violation
  FROM workspace.sdwa_gold.fact_violations
  WHERE system_is_active
  GROUP BY county
)
SELECT
  s.county,
  s.active_systems,
  s.community_population_served,
  COALESCE(v.violations, 0)                          AS violations,
  COALESCE(v.health_based_violations, 0)             AS health_based_violations,
  COALESCE(v.systems_with_health_based_violation, 0) AS systems_with_health_based_violation,
  ROUND(COALESCE(v.systems_with_health_based_violation, 0) / s.active_systems * 100, 1)
                                                     AS pct_systems_with_health_based_violation
FROM systems s
LEFT JOIN violations v ON s.county = v.county

In [0]:
%sql
SELECT
  (SELECT COUNT(*) FROM workspace.sdwa_gold.fact_violations)                        AS fact_rows,
  (SELECT COUNT(*) FROM workspace.sdwa_silver.violations
    WHERE noncompliance_begin_date BETWEEN '2021-01-01' AND '2025-12-31')          AS expected_rows,
  (SELECT COUNT(*) FROM workspace.sdwa_gold.dim_water_system)                       AS systems,
  (SELECT COUNT(*) FROM workspace.sdwa_gold.county_summary)                         AS counties